# 🌲 Semana 15 · Unidad 4 — Árboles 2-3 y árboles rojo-negro

## Información del Curso

| Aspecto | Detalle |
|--------|--------|
| **Universidad** | Universidad de Talca, Chile |
| **Carrera** | Ingeniería Civil en Informática |
| **Semestre** | 2°-3° año |
| **Curso** | Algoritmos y Estructuras de Datos |
| **Docente** | PhD. César Astudillo |
| **Clase** | Semana 15 · Unidad 4 — Árboles balanceados |
| **Duración** | 90 minutos |

---
> 🎯 *Este notebook está diseñado para ser ejecutado en clase de forma interactiva.*
> *Ejecuta las celdas en orden de arriba hacia abajo.*

In [ ]:
# Verificación de dependencias — ejecutar primero
import sys
required = {
    'numpy': 'numpy',
    'matplotlib': 'matplotlib',
}
for nombre, paquete in required.items():
    try:
        __import__(paquete)
        print(f"✅ {nombre} instalado correctamente")
    except ImportError:
        print(f"❌ {nombre} NO encontrado — instala con: pip install {paquete}")
print("\n🐍 Python", sys.version.split()[0], "| Todo listo para comenzar.")

## 🎯 Objetivos de Aprendizaje

Al finalizar esta sesión, el estudiante será capaz de:

1. **Comprender** por qué un BST simple degenera a $O(n)$ y por qué eso no es un caso raro
   de laboratorio sino el escenario más común en la práctica.
2. **Identificar** la estructura de un árbol 2-3: nodos de 2 y 3 claves, y la inserción por
   división ascendente que mantiene todas las hojas al mismo nivel.
3. **Implementar** las tres operaciones elementales de un árbol rojo-negro —
   `rotar_izquierda`, `rotar_derecha` y `cambiar_colores` — y explicar qué invariante
   restablece cada una.
4. **Analizar** por qué la altura de un árbol rojo-negro está acotada por $2\log_2 n$ y qué
   garantía da eso sobre `get`, `put` y `delete`.
5. **Comparar** BST simple, árbol balanceado y tabla hash como implementaciones de un
   diccionario, y justificar cuál conviene según el caso de uso.

# Sección 1: El problema que dejamos abierto (12 minutos)

## ¿Dónde quedamos?

La semana pasada implementamos el **BST** y medimos su altura. La conclusión fue incómoda:

| Orden de inserción | Altura resultante | Costo de `get` |
|---|---|---|
| Aleatorio | $\approx 1{,}39 \log_2 n$ | $O(\log n)$ |
| **Ordenado** | $n - 1$ | $O(n)$ |

Un BST construido con claves ya ordenadas **no es un árbol**: es una lista enlazada con
sintaxis de árbol.

## Y esto no es un caso de laboratorio

> ⚠️ **Importante:** las claves ordenadas son la norma, no la excepción.
> Piensa en insertar registros por *timestamp*, por número de factura, por RUT correlativo,
> o en recargar un índice desde un archivo que ya venía ordenado. En todos esos casos el
> BST simple colapsa justo cuando más datos tiene.

> 🎙️ **[PAUSA PROFESOR]** Pregunta sugerida: "¿Se les ocurre una forma de evitarlo sin
> cambiar la estructura? ¿Y si desordenamos las claves antes de insertarlas — qué problema
> tiene esa idea en un sistema que recibe datos en línea?"

Ejecutemos la evidencia una vez más.

In [ ]:
import random
random.seed(2026)

class NodoBST:
    """Nodo de un BST simple, sin balanceo."""
    __slots__ = ("clave", "valor", "izq", "der")
    def __init__(self, clave, valor):
        self.clave, self.valor = clave, valor
        self.izq = self.der = None


def bst_insertar(raiz, clave, valor):
    """
    Inserta en un BST simple, SIN balanceo. Versión iterativa.

    Se implementa de forma iterativa a propósito: con claves ordenadas este
    árbol degenera a una lista de altura n-1, y una versión recursiva
    reventaría el límite de recursión de Python antes de que alcancemos a
    mostrar el problema. El punto de la clase es justamente ese.

    Complejidad:
        Temporal: O(altura) — que en el peor caso es O(n)
        Espacial: O(1)
    """
    if raiz is None:
        return NodoBST(clave, valor)
    x = raiz
    while True:
        if clave < x.clave:
            if x.izq is None:
                x.izq = NodoBST(clave, valor)
                return raiz
            x = x.izq
        elif clave > x.clave:
            if x.der is None:
                x.der = NodoBST(clave, valor)
                return raiz
            x = x.der
        else:
            x.valor = valor
            return raiz


def altura(raiz):
    """Altura del árbol en número de aristas. Árbol vacío = -1."""
    pila, h = [(raiz, 0)], -1
    while pila:
        nodo, d = pila.pop()
        if nodo is None:
            continue
        h = max(h, d)
        pila.append((nodo.izq, d + 1))
        pila.append((nodo.der, d + 1))
    return h


import math
print(f"{'n':>7} {'aleatorio':>12} {'ordenado':>12} {'ideal log2(n)':>15}")
print("-" * 48)
for n in [100, 500, 1000, 2000, 4000]:
    claves = list(range(n))

    r_ale = None
    mezcladas = claves[:]
    random.shuffle(mezcladas)
    for k in mezcladas:
        r_ale = bst_insertar(r_ale, k, k)

    r_ord = None
    for k in claves:
        r_ord = bst_insertar(r_ord, k, k)

    print(f"{n:>7} {altura(r_ale):>12} {altura(r_ord):>12} {math.log2(n):>15.1f}")

print("\n👉 La columna 'ordenado' es exactamente n-1: el árbol es una lista enlazada.")

## La pregunta de hoy

> 📌 **El objetivo:** una estructura de diccionario donde `get`, `put` y `delete` sean
> $O(\log n)$ **en el peor caso**, sin depender del orden en que lleguen las claves.

La idea que lo consigue tiene dos presentaciones. Vamos a ver las dos, porque la primera
es la que se entiende y la segunda es la que se implementa:

1. **Árbol 2-3** — la idea conceptual. Fácil de razonar, incómodo de programar.
2. **Árbol rojo-negro** — la misma idea codificada en un árbol binario. Feo de razonar,
   directo de programar.

# Sección 2: Árboles 2-3 (20 minutos)

## La idea: dejar que un nodo crezca

En un BST, cuando insertamos una clave creamos un nodo nuevo **abajo**, y por eso el árbol
se alarga hacia un lado. El árbol 2-3 hace lo contrario: cuando un nodo recibe una clave
nueva, primero **engorda**, y solo se parte cuando ya no le cabe.

> 📌 **Definición:** un **árbol 2-3** es un árbol de búsqueda donde cada nodo interno es:
> - un **2-nodo**: 1 clave y 2 hijos, o
> - un **3-nodo**: 2 claves y 3 hijos,
>
> y **todas las hojas están exactamente a la misma profundidad**.

```
        2-nodo                         3-nodo
         (M)                          (E  J)
        /   \                        /   |   \
      <M     >M                    <E  E..J   >J
```

## Inserción: engordar y, si desborda, dividir hacia arriba

El algoritmo tiene un solo caso interesante:

1. Busca la hoja donde correspondería la clave.
2. **Si la hoja es un 2-nodo**, se convierte en 3-nodo. Listo — el árbol no cambia de altura.
3. **Si la hoja es un 3-nodo**, se convertiría en un "4-nodo" temporal de 3 claves. Eso no
   está permitido, así que **se divide**: la clave del medio **sube al padre** y las otras
   dos quedan como dos 2-nodos.
4. Si al subir la clave el padre también desborda, se repite el proceso hacia arriba.
5. Si el desborde llega hasta la raíz, la raíz se divide y **el árbol crece un nivel**.

> 💡 **Insight:** aquí está todo el truco. Un árbol 2-3 **nunca crece por abajo, solo por
> arriba**, y cuando crece lo hace de forma pareja para todas las hojas a la vez. Por eso
> es imposible que una rama quede más larga que otra.

## Traza: insertar S, E, A, R, C, H en un árbol 2-3

```
insertar S        (S)                        un 2-nodo

insertar E        (E S)                      el 2-nodo engorda a 3-nodo

insertar A        (A E S)  -> desborda
                  sube la clave del medio:
                       (E)
                      /   \
                    (A)   (S)                el árbol crece un nivel

insertar R             (E)
                      /   \
                    (A)   (R S)              la hoja engorda

insertar C             (E)
                      /   \
                  (A C)   (R S)              la otra hoja engorda

insertar H        (A C) queda igual; H va a (R S) -> (H R S) desborda
                  sube R al padre:  (E R)
                       (E R)
                      /  |  \
                   (A C)(H)(S)                el padre absorbe, sin crecer
```

> ⚠️ **Importante:** en ningún momento una hoja quedó más profunda que otra. Esa es la
> propiedad que da la garantía.

> 🎙️ **[PAUSA PROFESOR]** Pregunta sugerida: "Inserten ustedes la X y la M. ¿En cuál de
> las dos crece el árbol?\"

## La garantía de altura

Sea $h$ la altura de un árbol 2-3 con $n$ claves.

- **Peor caso (todos 2-nodos):** el árbol es un árbol binario completo, así que
  $n \ge 2^{h+1} - 1$, de donde $h \le \log_2 n$.
- **Mejor caso (todos 3-nodos):** cada nivel se ramifica en 3, así que
  $n \ge 2(3^{h+1}-1)/2$, de donde $h \ge \log_3 n$.

$$\log_3 n \;\le\; h \;\le\; \log_2 n$$

Para $n = 10^9$: la altura está entre **19 y 30**. Comparado con los $10^9 - 1$ niveles del
BST degenerado, la diferencia no es de constante, es de categoría.

> 📌 **Conclusión:** `get`, `put` y `delete` son $O(\log n)$ **en el peor caso**, sin
> supuestos sobre el orden de llegada de las claves.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n = np.logspace(1, 9, 200)
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(n, n - 1,          linewidth=2, label="BST degenerado — O(n)")
ax.plot(n, np.log2(n),     linewidth=2, label="cota superior 2-3: log₂ n")
ax.plot(n, np.log(n)/np.log(3), linewidth=2, label="cota inferior 2-3: log₃ n")
ax.fill_between(n, np.log(n)/np.log(3), np.log2(n), alpha=0.25,
                label="altura garantizada de un árbol 2-3")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Número de claves (n)")
ax.set_ylabel("Altura del árbol")
ax.set_title("El árbol 2-3 vive en una franja estrecha; el BST degenerado no tiene techo")
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

# Sección 3: Árboles rojo-negro (20 minutos)

## El problema práctico del árbol 2-3

Conceptualmente el 2-3 es claro. Programarlo directamente es tedioso: hay dos tipos de nodo,
la división se propaga hacia arriba, y el código termina lleno de casos especiales.

La solución de Guibas y Sedgewick (1978) es elegante: **representar el árbol 2-3 con un
árbol binario normal**, marcando con color los enlaces que "pegan" dos nodos que en realidad
son un 3-nodo.

## El truco de la codificación

> 📌 **Definición:** un **enlace rojo** une dos nodos que juntos representan un 3-nodo.
> Un **enlace negro** es un enlace normal del árbol 2-3.

```
   3-nodo del árbol 2-3          su representación rojo-negro
                                   (inclinada a la izquierda)

          (E  J)                          J
         /   |   \                       /  \
       <E  E..J  >J                    E ← enlace ROJO
                                      / \
                                    <E  E..J      >J
```

Usamos la variante **left-leaning** (LLRB): los enlaces rojos siempre apuntan a la
izquierda. Eso reduce los casos a manejar de seis a tres.

## Los tres invariantes

Un árbol rojo-negro left-leaning cumple, en todo momento:

1. **Inclinación:** los enlaces rojos siempre se inclinan a la **izquierda**.
2. **No consecutivos:** ningún nodo tiene dos enlaces rojos seguidos hacia abajo.
3. **Balance negro perfecto:** todo camino de la raíz a una hoja tiene el **mismo número de
   enlaces negros**.

> 💡 **Insight:** el invariante 3 es exactamente la propiedad "todas las hojas al mismo
> nivel" del árbol 2-3. Los enlaces rojos no cuentan porque no son aristas reales del 2-3:
> son pegamento dentro de un nodo.

> 🎙️ **[PAUSA PROFESOR]** Pregunta sugerida: "Si borro mentalmente todos los enlaces rojos
> y colapso los nodos que unían, ¿qué obtengo?\"

In [ ]:
ROJO, NEGRO = True, False

class Nodo:
    """
    Nodo de un árbol rojo-negro left-leaning (LLRB).

    El color guardado es el del ENLACE QUE ENTRA a este nodo desde su padre,
    no el del nodo en sí. Esa convención es la que hace el código corto.
    """
    __slots__ = ("clave", "valor", "izq", "der", "color", "n")

    def __init__(self, clave, valor, color=ROJO):
        self.clave, self.valor = clave, valor
        self.izq = self.der = None
        self.color = color      # color del enlace padre -> este nodo
        self.n = 1              # tamaño del subárbol


def es_rojo(nodo) -> bool:
    """
    Indica si el enlace que entra a `nodo` es rojo.

    Un enlace nulo se considera NEGRO por convención: así el invariante de
    balance negro se cumple trivialmente en las hojas.

    Complejidad: O(1)
    """
    if nodo is None:
        return False
    return nodo.color == ROJO


def tam(nodo) -> int:
    """Tamaño del subárbol; 0 si es nulo. Complejidad: O(1)"""
    return 0 if nodo is None else nodo.n


print("✅ Estructura de nodo definida.")
print("   Recuerda: el color vive en el ENLACE, no en el nodo.")

# Sección 4: Las tres operaciones de rebalanceo (20 minutos)

Todo el balanceo se consigue con tres operaciones locales, cada una de costo $O(1)$.
No hay más. Cualquier árbol rojo-negro se mantiene con estas tres.

## 4.1 `rotar_izquierda` — enderezar un rojo que quedó a la derecha

Corrige el invariante 1 (inclinación).

```
     h                        x
    / \                      / \
   A   x  ← ROJO   ==>  ROJO→h   C
      / \                  / \
     B   C                A   B
```

## 4.2 `rotar_derecha` — preparar una división

Es la inversa. Se usa cuando hay dos rojos seguidos a la izquierda.

```
       h                    x
      / \                  / \
ROJO→x   C    ==>         A   h ←ROJO
    / \                      / \
   A   B                    B   C
```

## 4.3 `cambiar_colores` — la división del 4-nodo

Cuando un nodo tiene **ambos** hijos con enlace rojo, eso representa un 4-nodo temporal.
Dividirlo es simplemente cambiar tres colores: los hijos pasan a negro y el enlace al padre
pasa a rojo. **Eso es exactamente "la clave del medio sube al padre"** del árbol 2-3.

```
        h  ← negro                  h  ← ROJO (sube al padre)
      /   \                       /   \
 ROJO→a   b ←ROJO      ==>      a       b   ← ambos negros
```

> 💡 **Insight:** `cambiar_colores` es la división del árbol 2-3, y es $O(1)$. La propagación
> hacia arriba ocurre sola, porque el rebalanceo se aplica al volver de la recursión.

> 🎙️ **[PAUSA PROFESOR]** Pregunta sugerida: "¿Por qué `cambiar_colores` no rompe el
> invariante 3 (balance negro)?\"

In [ ]:
def rotar_izquierda(h: "Nodo") -> "Nodo":
    """
    Endereza un enlace rojo que apunta a la derecha, dejándolo a la izquierda.

    Parámetros:
        h (Nodo): nodo cuyo hijo derecho tiene enlace rojo

    Retorna:
        Nodo: la nueva raíz del subárbol

    Complejidad:
        Temporal: O(1) — solo reasigna punteros
        Espacial: O(1)
    """
    x = h.der
    h.der = x.izq
    x.izq = h
    x.color = h.color      # x hereda el color del enlace que entraba a h
    h.color = ROJO         # y h queda colgando de un enlace rojo
    x.n = h.n
    h.n = 1 + tam(h.izq) + tam(h.der)
    return x


def rotar_derecha(h: "Nodo") -> "Nodo":
    """
    Inversa de rotar_izquierda: pasa un enlace rojo de la izquierda a la derecha.

    Parámetros:
        h (Nodo): nodo cuyo hijo izquierdo tiene enlace rojo

    Retorna:
        Nodo: la nueva raíz del subárbol

    Complejidad:
        Temporal: O(1)
        Espacial: O(1)
    """
    x = h.izq
    h.izq = x.der
    x.der = h
    x.color = h.color
    h.color = ROJO
    x.n = h.n
    h.n = 1 + tam(h.izq) + tam(h.der)
    return x


def cambiar_colores(h: "Nodo") -> None:
    """
    Divide un 4-nodo temporal: los dos hijos pasan a negro y h pasa a rojo.

    Es la traducción exacta de «la clave del medio sube al padre» del árbol 2-3.

    Parámetros:
        h (Nodo): nodo con AMBOS hijos en rojo

    Retorna:
        None — modifica los colores in-place.

    Complejidad:
        Temporal: O(1)
        Espacial: O(1)
    """
    h.color = ROJO
    h.izq.color = NEGRO
    h.der.color = NEGRO


print("✅ Las tres operaciones de rebalanceo están definidas.")

## Poniéndolas juntas: `put`

Lo notable es que la inserción es idéntica a la de un BST, **más tres líneas** al volver de
la recursión. Esas tres líneas restablecen los invariantes en el camino de vuelta hacia la
raíz.

In [ ]:
class ArbolRojoNegro:
    """
    Symbol table ordenada implementada como árbol rojo-negro left-leaning.

    Garantiza O(log n) en el PEOR caso para get, put y contains, sin
    supuestos sobre el orden de inserción de las claves.
    """

    def __init__(self):
        self.raiz = None

    # ── búsqueda: idéntica a un BST ────────────────────────────────────────
    def get(self, clave):
        """
        Busca el valor asociado a la clave. Retorna None si no está.

        Complejidad:
            Temporal: O(log n) en el peor caso — la altura está acotada
            Espacial: O(1) — iterativo
        """
        x = self.raiz
        while x is not None:
            if clave < x.clave:
                x = x.izq
            elif clave > x.clave:
                x = x.der
            else:
                return x.valor
        return None

    def __contains__(self, clave):
        return self.get(clave) is not None

    def __len__(self):
        return tam(self.raiz)

    # ── inserción ─────────────────────────────────────────────────────────
    def put(self, clave, valor):
        """
        Inserta o actualiza una clave.

        Complejidad:
            Temporal: O(log n) en el peor caso
            Espacial: O(log n) por la pila de recursión
        """
        self.raiz = self._put(self.raiz, clave, valor)
        self.raiz.color = NEGRO      # la raíz siempre es negra

    def _put(self, h, clave, valor):
        if h is None:
            return Nodo(clave, valor, ROJO)   # los nodos nuevos entran en ROJO

        # ── parte idéntica a un BST ──
        if clave < h.clave:
            h.izq = self._put(h.izq, clave, valor)
        elif clave > h.clave:
            h.der = self._put(h.der, clave, valor)
        else:
            h.valor = valor

        # ── las tres líneas que hacen todo el balanceo ──
        if es_rojo(h.der) and not es_rojo(h.izq):
            h = rotar_izquierda(h)                  # invariante 1: inclinación
        if es_rojo(h.izq) and es_rojo(h.izq.izq):
            h = rotar_derecha(h)                    # invariante 2: dos rojos seguidos
        if es_rojo(h.izq) and es_rojo(h.der):
            cambiar_colores(h)                      # división del 4-nodo

        h.n = 1 + tam(h.izq) + tam(h.der)
        return h

    # ── utilidades de diagnóstico ─────────────────────────────────────────
    def altura(self):
        """Altura en aristas, contando enlaces rojos y negros."""
        def _h(x):
            return -1 if x is None else 1 + max(_h(x.izq), _h(x.der))
        return _h(self.raiz)

    def altura_negra(self):
        """Número de enlaces negros de la raíz a una hoja."""
        h, x = 0, self.raiz
        while x is not None:
            if not es_rojo(x):
                h += 1
            x = x.izq
        return h

    def es_valido(self):
        """Verifica los tres invariantes. Retorna (bool, mensaje)."""
        def chequear(x, negros, esperado):
            if x is None:
                return negros == esperado, "balance negro roto"
            if es_rojo(x.der):
                return False, f"enlace rojo a la derecha en {x.clave!r}"
            if es_rojo(x) and es_rojo(x.izq):
                return False, f"dos enlaces rojos seguidos en {x.clave!r}"
            sig = negros + (0 if es_rojo(x) else 1)
            ok, msg = chequear(x.izq, sig, esperado)
            if not ok:
                return False, msg
            return chequear(x.der, sig, esperado)

        if self.raiz is None:
            return True, "árbol vacío"
        if es_rojo(self.raiz):
            return False, "la raíz debe ser negra"
        return chequear(self.raiz, 0, self.altura_negra())


print("✅ ArbolRojoNegro definido.")

In [ ]:
# La prueba del algodón: insertar en ORDEN CRECIENTE, el peor caso del BST
print("Insertando 0..n-1 en orden creciente — el caso que destruye al BST simple\n")
print(f"{'n':>8} {'altura BST':>12} {'altura RN':>11} {'2·log₂n':>10} {'¿válido?':>10}")
print("-" * 56)

for n in [100, 500, 1000, 5000, 20000]:
    arn = ArbolRojoNegro()
    for k in range(n):
        arn.put(k, k)

    # Construir el BST degenerado cuesta O(n²), así que solo lo MEDIMOS hasta
    # n=3000; para n mayores mostramos el valor teórico n-1, marcado con *.
    if n <= 3000:
        r_ord = None
        for k in range(n):
            r_ord = bst_insertar(r_ord, k, k)
        h_bst = altura(r_ord)
        assert h_bst == n - 1, "el BST ordenado debería degenerar a una lista"
        etiqueta_bst = f"{h_bst}"
    else:
        etiqueta_bst = f"{n - 1}*"

    ok, msg = arn.es_valido()
    print(f"{n:>8} {etiqueta_bst:>12} {arn.altura():>11} {2*math.log2(n):>10.1f} {'✅' if ok else '❌ '+msg:>10}")

print("\n(*) valor teórico n-1; medido directamente hasta n=3000.")
print("\n👉 El BST crece como n. El rojo-negro se queda pegado bajo 2·log₂n,")
print("   con exactamente la misma secuencia de inserción.")

# Sección 5: La garantía y su demostración (10 minutos)

## ¿Por qué la altura es a lo más $2\log_2 n$?

El argumento es corto si ya entendiste la codificación:

1. Por el invariante 3, todo camino raíz→hoja tiene el mismo número de enlaces **negros**.
   Llamemos $b$ a ese número (la *altura negra*).
2. Si quitamos los enlaces rojos y colapsamos los 3-nodos, queda el árbol 2-3 subyacente,
   que tiene altura exactamente $b$. Por la Sección 2, $b \le \log_2 n$.
3. Por el invariante 2, **no hay dos enlaces rojos consecutivos**. Entonces, en cualquier
   camino, entre dos enlaces negros hay a lo más uno rojo: los rojos son a lo más tantos
   como los negros.
4. Por lo tanto la altura total cumple $h \le 2b \le 2\log_2 n$.

$$\boxed{h \le 2\log_2 n}$$

> 📌 **Consecuencia:** `get`, `put` y `delete` son $O(\log n)$ **en el peor caso**.
> No "en promedio", no "si las claves llegan desordenadas". Siempre.

## El costo que se paga

| | BST simple | Rojo-negro |
|---|---|---|
| `get` / `put` promedio | $O(\log n)$ | $O(\log n)$ |
| `get` / `put` **peor caso** | $O(n)$ | $O(\log n)$ |
| Memoria por nodo | clave, valor, 2 punteros | + **1 bit de color** |
| Complejidad del código | ~10 líneas | ~25 líneas |
| Mantiene el orden de las claves | sí | sí |

> 💡 **Insight:** un bit por nodo y tres líneas en `put` compran la eliminación completa del
> peor caso. Es de las mejores relaciones costo/beneficio de todo el curso.

> 🎙️ **[PAUSA PROFESOR]** Pregunta sugerida: "Si un rojo-negro garantiza O(log n) siempre,
> ¿por qué el `dict` de Python no es un árbol rojo-negro? ¿Qué gana renunciando al orden?\"

In [ ]:
# Comparación empírica de altura sobre las mismas claves
tam_n = [200, 400, 800, 1600, 3200]
alt_rn_ord, alt_rn_ale, alt_bst_ale, cota = [], [], [], []

for n in tam_n:
    a1 = ArbolRojoNegro()
    for k in range(n):
        a1.put(k, k)
    alt_rn_ord.append(a1.altura())

    mez = list(range(n))
    random.shuffle(mez)
    a2 = ArbolRojoNegro()
    for k in mez:
        a2.put(k, k)
    alt_rn_ale.append(a2.altura())

    r = None
    for k in mez:
        r = bst_insertar(r, k, k)
    alt_bst_ale.append(altura(r))

    cota.append(2 * math.log2(n))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(tam_n, alt_rn_ord,  "o-", linewidth=2, label="Rojo-negro, inserción ORDENADA")
ax.plot(tam_n, alt_rn_ale,  "s-", linewidth=2, label="Rojo-negro, inserción aleatoria")
ax.plot(tam_n, alt_bst_ale, "^-", linewidth=2, label="BST simple, inserción aleatoria")
ax.plot(tam_n, cota, "--", linewidth=2, color="red", label="cota teórica 2·log₂n")
ax.set_xlabel("Número de claves (n)")
ax.set_ylabel("Altura del árbol")
ax.set_title("El rojo-negro respeta la cota incluso en el peor orden de inserción")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Nota: el BST con inserción ORDENADA no aparece en el gráfico porque su altura")
print("es n-1 y aplastaría la escala. Para n=3200 sería 3199 frente a los ~14 del rojo-negro.")

# Sección 6: Explorador interactivo (5 minutos)

Construye un árbol rojo-negro y observa cómo la altura se mantiene acotada
independientemente del orden de inserción.

In [ ]:
# ── Explorador parametrizable ───────────────────────────────────────────────
# Cambia los valores de las llamadas de abajo y vuelve a ejecutar.

def explorar_arbol(n=64, orden="ordenada", verbose=True):
    """
    Construye un árbol rojo-negro con n claves y reporta su altura.

    Parámetros:
        n (int): número de claves
        orden (str): 'ordenada', 'inversa' o 'aleatoria'
        verbose (bool): imprime el detalle

    Retorna:
        (int, int): altura del árbol rojo-negro y del BST simple equivalente
    """
    if orden == "ordenada":
        claves = list(range(n))
    elif orden == "inversa":
        claves = list(range(n - 1, -1, -1))
    else:
        claves = list(range(n)); random.shuffle(claves)

    arbol = ArbolRojoNegro()
    for k in claves:
        arbol.put(k, k)
    ok, msg = arbol.es_valido()

    r = None
    for k in claves:
        r = bst_insertar(r, k, k)

    cota = 2 * math.log2(n) if n > 0 else 0
    if verbose:
        print(f"n = {n:<6} inserción {orden:<10} "
              f"rojo-negro h = {arbol.altura():<3} "
              f"BST h = {altura(r):<6} "
              f"cota 2·log₂n = {cota:>5.1f}   "
              f"{'✅' if ok and arbol.altura() <= cota else '❌ ' + msg}")
    return arbol.altura(), altura(r)


for orden in ("ordenada", "inversa", "aleatoria"):
    explorar_arbol(n=64, orden=orden)
print()
for n in (16, 128, 1024):
    explorar_arbol(n=n, orden="ordenada")

print("\n👉 El BST se dispara con inserción ordenada; el rojo-negro se queda bajo la cota")
print("   en los tres órdenes. Cambia n y el orden en las llamadas de arriba.")

## 🧪 Ejercicio 1: Contar enlaces rojos ⭐

**Descripción:** implementa una función que cuente cuántos enlaces rojos hay en un árbol
rojo-negro. Ese número te dice cuántos 3-nodos tiene el árbol 2-3 equivalente.

**Entrada:** la raíz de un árbol rojo-negro (un `Nodo` o `None`).
**Salida:** entero con la cantidad de nodos cuyo enlace de entrada es rojo.

**Ejemplo:**
```
Entrada: árbol construido con put(1), put(2), put(3)
Salida:  0     (queda perfectamente balanceado, sin 3-nodos)
```

**Restricciones:** no modifiques el árbol.
**Complejidad esperada:** O(n)

In [ ]:
def contar_rojos(raiz):
    """
    Cuenta los enlaces rojos del árbol.

    Parámetros:
        raiz: nodo raíz del árbol rojo-negro (o None)
    Retorna:
        int: cantidad de enlaces rojos
    """
    # Tu código aquí
    pass

In [ ]:
def verificar_ejercicio_1(fn):
    """Ejecuta casos de prueba para el ejercicio 1."""
    import time
    def construir(claves):
        a = ArbolRojoNegro()
        for k in claves:
            a.put(k, k)
        return a

    def rojos_referencia(x):
        if x is None:
            return 0
        return (1 if es_rojo(x) else 0) + rojos_referencia(x.izq) + rojos_referencia(x.der)

    entradas = [
        ([], "árbol vacío"),
        ([1], "un solo nodo"),
        ([1, 2, 3], "tres claves, queda balanceado"),
        (list(range(10)), "diez claves en orden"),
        (list(range(100)), "cien claves en orden"),
        ([5, 3, 8, 1, 4, 7, 9], "inserción arbitraria"),
    ]
    aprobados = 0
    for claves, desc in entradas:
        arbol = construir(claves)
        esperado = rojos_referencia(arbol.raiz)
        t0 = time.perf_counter()
        try:
            resultado = fn(arbol.raiz)
            t1 = time.perf_counter()
            if resultado == esperado:
                print(f"  ✅ {desc} ({(t1-t0)*1000:.2f}ms) → {resultado} enlaces rojos")
                aprobados += 1
            else:
                print(f"  ❌ {desc}")
                print(f"     Esperado: {esperado}")
                print(f"     Obtenido: {resultado}")
        except Exception as e:
            print(f"  💥 {desc} — Error: {e}")
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados == len(entradas) else f'⚠️  {aprobados}/{len(entradas)} casos correctos'}")

verificar_ejercicio_1(contar_rojos)

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# def contar_rojos(raiz):
#     """Recorre el árbol sumando 1 por cada enlace rojo encontrado."""
#     # Paso 1: caso base, un enlace nulo no aporta
#     if raiz is None:
#         return 0
#     # Paso 2: este nodo aporta 1 si su enlace de entrada es rojo
#     aqui = 1 if es_rojo(raiz) else 0
#     # Paso 3: sumar recursivamente ambos subárboles
#     return aqui + contar_rojos(raiz.izq) + contar_rojos(raiz.der)
#     # Complejidad: O(n) temporal, O(altura) espacial por la recursión

## 🧪 Ejercicio 2: Verificar el balance negro ⭐⭐

**Descripción:** implementa una función que verifique el **invariante 3**: todos los caminos
de la raíz a un enlace nulo tienen el mismo número de enlaces negros.

Es el invariante que traduce "todas las hojas al mismo nivel" del árbol 2-3, y el más
difícil de comprobar a ojo.

**Entrada:** la raíz de un árbol (un `Nodo` o `None`).
**Salida:** `True` si el balance negro se cumple, `False` si no.

**Ejemplo:**
```
Entrada: cualquier árbol construido con ArbolRojoNegro.put
Salida:  True
```

**Restricciones:** debes recorrer todos los caminos, no solo el de más a la izquierda.
**Complejidad esperada:** O(n)

> 💡 **Pista:** una función recursiva que retorne la altura negra del subárbol, o `-1` si
> detecta un desbalance, resuelve esto en una pasada.

In [ ]:
def balance_negro_ok(raiz):
    """
    Verifica que todos los caminos raíz→nulo tengan igual cantidad de enlaces negros.

    Parámetros:
        raiz: nodo raíz del árbol (o None)
    Retorna:
        bool: True si el invariante se cumple
    """
    # Tu código aquí
    pass

In [ ]:
def verificar_ejercicio_2(fn):
    """Ejecuta casos de prueba para el ejercicio 2, incluidos árboles CORRUPTOS."""
    import time

    def construir(claves):
        a = ArbolRojoNegro()
        for k in claves:
            a.put(k, k)
        return a.raiz

    # Árbol deliberadamente roto: una rama con un negro de más.
    roto = Nodo(10, 10, NEGRO)
    roto.izq = Nodo(5, 5, NEGRO)
    roto.der = Nodo(15, 15, NEGRO)
    roto.izq.izq = Nodo(2, 2, NEGRO)     # rompe el balance negro

    casos = [
        (None, True, "árbol vacío"),
        (construir([1]), True, "un nodo"),
        (construir([1, 2, 3]), True, "tres claves"),
        (construir(list(range(50))), True, "50 claves en orden"),
        (construir([5, 3, 8, 1, 4, 7, 9, 2, 6]), True, "inserción arbitraria"),
        (roto, False, "árbol CORRUPTO: rama con un negro de más"),
    ]
    aprobados = 0
    for raiz, esperado, desc in casos:
        t0 = time.perf_counter()
        try:
            resultado = fn(raiz)
            t1 = time.perf_counter()
            if bool(resultado) == esperado:
                print(f"  ✅ {desc} ({(t1-t0)*1000:.2f}ms)")
                aprobados += 1
            else:
                print(f"  ❌ {desc}")
                print(f"     Esperado: {esperado}")
                print(f"     Obtenido: {resultado}")
        except Exception as e:
            print(f"  💥 {desc} — Error: {e}")
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados == len(casos) else f'⚠️  {aprobados}/{len(casos)} casos correctos'}")

verificar_ejercicio_2(balance_negro_ok)

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# def balance_negro_ok(raiz):
#     """Retorna True si todos los caminos tienen igual número de enlaces negros."""
#     # Paso 1: función auxiliar que devuelve la altura negra, o -1 si hay desbalance
#     def altura_negra(x):
#         if x is None:
#             return 0                      # un enlace nulo cuenta como camino válido
#         iz = altura_negra(x.izq)
#         if iz == -1:
#             return -1                     # el desbalance se propaga hacia arriba
#         de = altura_negra(x.der)
#         if de == -1 or iz != de:
#             return -1                     # las dos ramas no coinciden
#         # Paso 2: sumar 1 solo si el enlace que entra a x es NEGRO
#         return iz + (0 if es_rojo(x) else 1)
#     # Paso 3: hay balance si la auxiliar no reportó -1
#     return altura_negra(raiz) != -1
#     # Complejidad: O(n) temporal, O(altura) espacial

## 🔬 Zona de Experimentación

Las siguientes celdas son tuyas para experimentar. Algunas sugerencias:
- ¿Qué pasa si comentas la línea de `rotar_izquierda` en `_put`? ¿Qué invariante se rompe
  primero, y para qué `n` lo notas?
- ¿Puedes construir una secuencia de inserción que haga la altura del rojo-negro lo más
  grande posible? ¿Qué tan cerca de $2\log_2 n$ logras llegar?
- Compara el tiempo de `put` de `ArbolRojoNegro` contra el `dict` nativo de Python.

In [ ]:
# Espacio libre para experimentar
# Sugerencia: rompe un invariante a propósito y usa es_valido() para ver qué reporta.


In [ ]:
# Espacio libre para experimentar
# Sugerencia: mide el tiempo de 100.000 inserciones en el árbol vs. en un dict.


## ✍️ Autoevaluación

Responde antes de la Prueba de la Unidad 4.

## ✍️ Autoevaluación

Responde cada pregunta antes de abrir la respuesta.

**1. ¿Cuál es la altura máxima de un árbol rojo-negro con n claves?**

- a) log₂ n
- b) 1,39 · log₂ n
- c) 2 · log₂ n
- d) n − 1

<details>
<summary>Ver respuesta</summary>

**c) 2 · log₂ n** — La altura negra es a lo más log₂n, y como no hay dos rojos seguidos, los enlaces rojos no superan a los negros: h ≤ 2·log₂n.

</details>

---

**2. En un árbol rojo-negro left-leaning, ¿qué representa un enlace ROJO?**

- a) Un nodo que hay que eliminar
- b) El pegamento entre dos nodos que juntos forman un 3-nodo del árbol 2-3
- c) Una rama desbalanceada que falta rotar
- d) El camino más largo de la raíz a una hoja

<details>
<summary>Ver respuesta</summary>

**b) El pegamento entre dos nodos que juntos forman un 3-nodo del árbol 2-3** — El enlace rojo no es una arista real del árbol 2-3: une las dos claves de un mismo 3-nodo. Por eso no cuenta en el balance negro.

</details>

---

**3. ¿A qué operación del árbol 2-3 corresponde `cambiar_colores`?**

- a) A la búsqueda de una clave
- b) A convertir un 2-nodo en 3-nodo
- c) A dividir un 4-nodo temporal, subiendo la clave del medio al padre
- d) A eliminar la clave mínima

<details>
<summary>Ver respuesta</summary>

**c) A dividir un 4-nodo temporal, subiendo la clave del medio al padre** — cambiar_colores pinta los dos hijos de negro y el nodo de rojo: eso es exactamente la división, y el 'rojo hacia arriba' es la clave del medio subiendo al padre.

</details>

---

**4. Insertamos las claves 1,2,3,…,10000 EN ORDEN en un BST simple y en un rojo-negro. ¿Qué alturas esperamos?**

- a) Ambas ≈ 14
- b) BST ≈ 14 y rojo-negro ≈ 9999
- c) BST = 9999 y rojo-negro ≈ 14 o menos
- d) Ambas = 9999

<details>
<summary>Ver respuesta</summary>

**c) BST = 9999 y rojo-negro ≈ 14 o menos** — El BST degenera a una lista enlazada (altura n−1 = 9999). El rojo-negro rebalancea en cada inserción y se queda bajo 2·log₂(10000) ≈ 26.

</details>

---

**5. ¿Cuál es el costo de rotar_izquierda?**

- a) O(1)
- b) O(log n)
- c) O(n)
- d) O(n log n)

<details>
<summary>Ver respuesta</summary>

**a) O(1)** — Solo reasigna un puñado de punteros y colores. El costo O(log n) de put viene de recorrer la altura, no de las rotaciones.

</details>

# Resumen

## Lo que aprendimos hoy

1. El BST simple **degenera a $O(n)$** con claves ordenadas, y las claves ordenadas son el
   caso común, no el raro.
2. El **árbol 2-3** resuelve el problema creciendo solo por arriba: todas las hojas quedan
   siempre al mismo nivel, y la altura queda entre $\log_3 n$ y $\log_2 n$.
3. El **árbol rojo-negro** es el árbol 2-3 codificado en un árbol binario, usando un bit de
   color por nodo para marcar los 3-nodos.
4. Todo el balanceo cabe en **tres operaciones $O(1)$**: `rotar_izquierda`, `rotar_derecha`
   y `cambiar_colores`, aplicadas al volver de la recursión de `put`.
5. El resultado es $h \le 2\log_2 n$ y, con eso, **$O(\log n)$ en el peor caso** para todas
   las operaciones.

## El problema que queda

El árbol rojo-negro nos da $O(\log n)$ garantizado **y** mantiene las claves ordenadas
(podemos hacer `min`, `max`, `rank`, `select`, recorrido en orden).

Pero si **no necesitamos el orden** y solo queremos `get` y `put`, ¿podemos hacerlo mejor
que $\log n$? La respuesta es sí, y es el tema de la próxima clase: **tablas hash**, con
costo esperado $O(1)$.

> ⚠️ **Recordatorio de evaluación.** La **Prueba de la Unidad 4** cubre Symbol Tables,
> Binary Search Trees y árboles balanceados — es decir, hasta esta sesión inclusive.
> Tablas Hash **no entra** en esa prueba; se evalúa en el Examen Opcional Acumulativo.

## 📚 Lecturas Recomendadas y Práctica

### Textbooks

| Libro | Edición | Capítulo | Tema |
|-------|---------|----------|------|
| Cormen et al. (CLRS) — *Introduction to Algorithms* | 4ª ed. | Cap. 13 | Árboles rojo-negro: rotaciones, inserción y eliminación |
| Cormen et al. (CLRS) — *Introduction to Algorithms* | 4ª ed. | Cap. 13.1 | Propiedades y cota de altura |
| Goodrich, Tamassia & Goldwasser (GTG) — *Data Structures and Algorithms in Python* | 1ª ed. | Cap. 11.2–11.5 | Árboles balanceados: AVL, splay, 2-3 y rojo-negro |
| Bhargava (Grok) — *Grokking Algorithms* | 2ª ed. | Cap. 8 | Intuición de árboles balanceados |
| Miller & Ranum (M&R) — *Problem Solving with Algorithms and Data Structures Using Python* | 2011 | Cap. 6 | Árboles de búsqueda y su balanceo |

### Recursos gratuitos en línea

- 🌐 [VisuAlgo — Binary Search Tree / AVL](https://visualgo.net/en/bst) — inserción y rotaciones animadas paso a paso.
- 🌐 [Red/Black Tree Visualization (USFCA)](https://www.cs.usfca.edu/~galles/visualization/RedBlack.html) — construye el árbol clave por clave y observa cada rotación.
- 📄 [Left-Leaning Red-Black Trees — Robert Sedgewick (Princeton)](https://sedgewick.io/wp-content/themes/sedgewick/papers/2008LLRB.pdf) — el artículo original de la variante que implementamos hoy.

### Práctica en Codeforces (soporta Python 3)

> 🔍 **Cómo filtrar:** ve a [codeforces.com/problemset](https://codeforces.com/problemset),
> escribe `data structures` o `trees` en **Tags** y ajusta **Rating**.

**Escala de dificultad orientativa para este curso:**

| Rating | Nivel | Descripción |
|--------|-------|-------------|
| 800 | ⭐ | Aplicación directa — la mayoría puede resolverlo |
| 1000–1100 | ⭐⭐ | Requiere una pequeña adaptación del concepto |
| 1200–1300 | ⭐⭐⭐ | Combina la idea con otra — desafío |

**Problemas recomendados para este tópico:**

| # | Problema | Rating | Por qué es útil |
|---|----------|--------|-----------------|
| 1 | [1213C — Book Reading](https://codeforces.com/problemset/problem/1213/C) | ⭐ 1200 | Consultas sobre un conjunto ordenado; motiva la estructura ordenada |
| 2 | [1005C — Summarize to the Power of Two](https://codeforces.com/problemset/problem/1005/C) | ⭐⭐ 1300 | Búsqueda de complementos: contrasta diccionario ordenado vs. hash |
| 3 | [1234D — Distinct Characters Queries](https://codeforces.com/problemset/problem/1234/D) | ⭐⭐⭐ 1600 | Consultas de rango con actualizaciones — el caso donde el orden importa |

⚠️ El problema 1 es el **mínimo esperado**. El 2 es intermedio y el 3 es desafío opcional.